# Kapitel 32.10 - JWT Refresh Token Rotation, Blacklist, Whitelist

In diesem Notebook behandeln wir den sicheren Lebenszyklus von Access- und Refresh-Tokens.

# Lernziele

- Rollen von Access- und Refresh-Token sauber trennen
- die Logik der Refresh-Rotation aufbauen
- Blacklist/Whitelist-Ansatz anwenden

# Voraussetzungen

- Grundlagen der Authentifizierung
- Kapitel 32.7

# Theorie

Das Modell aus kurzlebigem Access-Token und langlebigem Refresh-Token balanciert Sicherheit und Nutzererlebnis.

# Erklaerung

Rotation: Beim Einsatz eines Refresh-Tokens wird das alte Token ungueltig und ein neues Paar erzeugt.
Dieser Mechanismus reduziert das Risiko von Token-Replay-Angriffen.

# Syntax

```python
pair = token_service.issue_pair(user_id)
new_pair = token_service.rotate_refresh(old_refresh)
```

# Merke

- Refresh-Token niemals im Klartext loggen.
- Einen Revoke-Endpoint als Pflicht betrachten.

# Parameter

- exp-Laufzeit
- jti
- token type

# Rueckgabewert

Token pair: access_token + refresh_token

In [ ]:
# Beispiel 1: Token-Paar-Konzept
pair = {'access_token': 'a.b.c', 'refresh_token': 'r.x.y'}
print(pair.keys())

In [ ]:
# Beispiel 2: Whitelist-/Blacklist-Idee
refresh_whitelist = set(['jti-1'])
refresh_blacklist = set()

old_jti = 'jti-1'
refresh_whitelist.discard(old_jti)
refresh_blacklist.add(old_jti)
print(refresh_whitelist, refresh_blacklist)

In [ ]:
# Beispiel 3: Ergebnis der Rotation
new_jti = 'jti-2'
refresh_whitelist.add(new_jti)
print('aktiver Refresh-JTI:', refresh_whitelist)

# Praxisbeispiel

Token-Service des Referenzprojekts:
32_Architektur_und_Security_Patterns/projects/fastapi_clean_cqrs_jwt_sample/app/security/token_service.py

# Haeufige Fehler

1. Keine Refresh-Rotation vorhanden
2. Nach Revoke bleibt das Token faelschlich gueltig
3. Mit Access-Token den Refresh-Endpoint aufrufen

# Best Practice

- Token type validation
- JTI-Nachverfolgung
- Bereinigung der Revocation-Liste

# Tipp

Security-Ereignisse (fehlgeschlagene Verifikation, Nutzung widerrufener Tokens) auditieren.

# Uebung

Schreibe eine einfache rotate_refresh-Funktion und lehne die Wiederverwendung des alten Tokens ab.

In [ ]:
# Loesung
def rotate_refresh(jti, whitelist, blacklist):
    if jti in blacklist:
        return {'ok': False, 'error': 'revoked'}
    if jti not in whitelist:
        return {'ok': False, 'error': 'not whitelisted'}
    whitelist.remove(jti)
    blacklist.add(jti)
    new_jti = jti + '-new'
    whitelist.add(new_jti)
    return {'ok': True, 'new_jti': new_jti}

print(rotate_refresh('x', set(['x']), set()))

# Zusammenfassung

JWT plus Refresh-Rotation erhoeht die Sicherheit, muss aber mit klaren Lebenszyklusregeln umgesetzt werden.

# Weiterfuehrende Links

- OWASP JWT Cheat Sheet
- Token revocation patterns

## Technischer Tiefgang

Auf fortgeschrittenem Niveau steht nicht nur Funktionalitaet, sondern die technische Nachvollziehbarkeit im Vordergrund.
Begriffe, Risiken, Betriebsaspekte und Qualitaetskriterien werden explizit gemacht, damit Entscheidungen reproduzierbar bleiben.

## Zentrale Fachbegriffe

Design Constraint
Trade-off
Failure Mode
Observability Signal
Quality Gate
Regression Risk

In [ ]:
# Zusatzbeispiel: Priorisierung technischer Risiken
risks=[{"name":"regression","score":9},{"name":"operability","score":8},{"name":"complexity","score":7}]
for r in sorted(risks,key=lambda x:x["score"],reverse=True):
    print(r["name"], r["score"])

## Fallstudie (Praxis)

Praxis-Szenario: Ein Team liefert ein Feature aus, das lokal stabil wirkt, in Staging jedoch sporadisch ausfaellt.
Beschreibe ein strukturiertes Vorgehen mit Hypothesen, Messsignalen, Gegenbeweisen und finaler Ursachenbehebung.

## Haeufige Fehler und Debugging-Checkliste

- Sind Eingaben, Konfiguration und Randbedingungen explizit validiert?
- Sind reproduzierbare Schritte fuer den Fehler dokumentiert?
- Wurden Logs, Metriken und Tests gemeinsam ausgewertet?
- Ist die Korrektur durch einen neuen Test dauerhaft abgesichert?
- Gibt es eine kurze Lessons-Learned-Notiz fuer das Team?

## Pruefungsfragen und Kurzloesungen

1. Warum ist technische Reproduzierbarkeit fuer Qualitaet entscheidend?
Kurzloesung: Nur reproduzierbare Befunde lassen sich verifizieren, beheben und regressionssicher absichern.
2. Was ist ein typischer Fehler bei schnellen Feature-Releases?
Kurzloesung: Betriebs- und Risikoaspekte werden zu spaet betrachtet.
3. Welche Rolle hat ein Quality Gate?
Kurzloesung: Es verhindert unsichere Releases durch verbindliche Mindestkriterien.